# Giano — data analysis

Explore which station observations are available, how long gaps last, and how the station signal compares with its auxiliary data.

**Input:** the corrected NetCDF files in `data/2-processed-v2/`,This notebook reads them without changing files or training a model.
Change `PROCESSED_ROOT` only if the same dataset is stored elsewhere.


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from giano.netcdf import find_time_coord, open_dataset_robust

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "2-processed-v2"
MAX_POINTS_PER_FILE = 20_000  # bounded-memory distribution sample
RNG = np.random.default_rng(42)
if not PROCESSED_ROOT.is_dir():
    raise FileNotFoundError(f"Missing processed data: {PROCESSED_ROOT}")

## Inventory and data quality

Each row describes one station/variable file. Missingness counts unavailable hourly positions; it is not a model error.
The folders named `train`, `val` and `test` are **storage groups**. The model pools them and uses chronological 70/15/15 splits.

The processed `value` already applies quality filtering. The charts below show file counts by storage group and the fraction of missing positions per variable.


In [ ]:
records: list[dict[str, object]] = []
distribution_samples: dict[str, list[np.ndarray]] = {}
gap_lengths: dict[str, Counter[int]] = {}

for split in ("train", "val", "test"):
    for path in sorted((PROCESSED_ROOT / split).glob("*_merged.nc")):
        name_parts = path.stem.split("_")
        station = name_parts[0]
        variable = "_".join(name_parts[1:-1])
        with open_dataset_robust(path) as dataset:
            values = np.asarray(dataset["value"].to_numpy(), dtype=float).reshape(-1)
            time_name = find_time_coord(dataset)
            times = (
                pd.DatetimeIndex(dataset[time_name].values)
                if time_name
                else pd.DatetimeIndex([])
            )
            finite = np.isfinite(values)
            auxiliary = [
                str(name)
                for name in dataset.data_vars
                if name in {variable, "wind_u", "wind_v"}
            ]

        finite_values = values[finite]
        if finite_values.size > MAX_POINTS_PER_FILE:
            finite_values = RNG.choice(
                finite_values, MAX_POINTS_PER_FILE, replace=False
            )
        distribution_samples.setdefault(variable, []).append(finite_values)

        counter = gap_lengths.setdefault(variable, Counter())
        start = None
        for position, is_missing in enumerate(np.r_[~finite, False]):
            if is_missing and start is None:
                start = position
            elif not is_missing and start is not None:
                counter[position - start] += 1
                start = None

        records.append(
            {
                "split": split,
                "station": station,
                "variable": variable,
                "n_points": values.size,
                "n_observed": int(finite.sum()),
                "missing_pct": float((~finite).mean() * 100),
                "start": times.min() if len(times) else pd.NaT,
                "end": times.max() if len(times) else pd.NaT,
                "has_auxiliary": bool(auxiliary),
                "auxiliary_variables": ", ".join(auxiliary),
            }
        )

inventory = pd.DataFrame(records)
display(inventory.head())
display(
    inventory.groupby(["split", "variable"])
    .agg(
        files=("station", "size"),
        stations=("station", "nunique"),
        points=("n_points", "sum"),
        observed=("n_observed", "sum"),
    )
    .assign(missing_pct=lambda frame: (1 - frame.observed / frame.points) * 100)
    .round(2)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
inventory.groupby(["split", "variable"]).size().unstack(fill_value=0).plot.bar(
    ax=axes[0]
)
axes[0].set(title="Files by storage group and variable", ylabel="NetCDF files")
totals = inventory.groupby("variable")[["n_observed", "n_points"]].sum()
missing_by_variable = (
    100 * (1 - totals["n_observed"] / totals["n_points"])
).sort_values()
missing_by_variable.plot.bar(ax=axes[1], color="tab:red")
axes[1].set(title="Source missingness by variable", ylabel="Missing (%)")
plt.tight_layout()
plt.show()

## Distributions and natural gap lengths

Histograms use a bounded sample of observed values, with a separate axis for each variable's units. They describe the data, not reconstruction quality.
Gap lengths count consecutive missing hours in the source. A concentration of long gaps indicates less local context.


In [ ]:
variables = sorted(distribution_samples)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis, variable in zip(axes.flat, variables, strict=False):
    sample = np.concatenate(distribution_samples[variable])
    lower, upper = np.quantile(sample, [0.005, 0.995])
    axis.hist(sample[(sample >= lower) & (sample <= upper)], bins=60, alpha=0.8)
    axis.set_title(variable.replace("_", " ").title())
    axis.set_ylabel("Sampled observations")
for axis in axes.flat[len(variables) :]:
    axis.set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for axis, variable in zip(axes.flat, sorted(gap_lengths), strict=False):
    counts = gap_lengths[variable]
    lengths = np.array(sorted(counts))
    frequencies = np.array([counts[length] for length in lengths])
    axis.bar(np.minimum(lengths, 168), frequencies)
    axis.set_yscale("log")
    axis.set(
        title=variable.replace("_", " ").title(),
        xlabel="Gap length (hours; clipped at 168)",
        ylabel="Count",
    )
for axis in axes.flat[len(gap_lengths) :]:
    axis.set_visible(False)
plt.tight_layout()
plt.show()

## Inspect one station and its auxiliary signal

Choose a storage group, variable and number of days below; the example selects the first matching station.
The station trace is `value`. Auxiliary traces are reanalysis inputs.
For wind, `wind_u` and `wind_v` are vector components.


In [ ]:
SELECTED_SPLIT = "test"
SELECTED_VARIABLE = "temperature"
DAYS = 14

selected_path = sorted(
    (PROCESSED_ROOT / SELECTED_SPLIT).glob(f"*_{SELECTED_VARIABLE}_merged.nc")
)[0]
with open_dataset_robust(selected_path) as dataset:
    time_name = find_time_coord(dataset)
    frame = dataset.to_dataframe().reset_index().set_index(time_name)

columns = ["value"] + [
    column
    for column in frame.columns
    if column in {SELECTED_VARIABLE, "wind_u", "wind_v"}
]
frame[columns].iloc[: 24 * DAYS].plot(figsize=(15, 5), title=selected_path.name)
plt.ylabel("Value")
plt.grid(alpha=0.3)
plt.show()